# Inspection des donnees brutes

- **Objectif** : Avoir une idee sur les donnees avec lesquelles on va travailler, pour qu'on pouisse les transofrmer en un seul jeu de donnees avec une variable cible (Churn), et des variables explicatives (features).

In [1]:
#Importation des libraries necessaires
import numpy as np
import pandas as pd
import seaborn as sb 
import matplotlib.pyplot as plt
from config import (
    RAW_INSCRIPTIONS,
    RAW_EVALUATIONS,
    RAW_ACTIVITES,
    RAW_PROGRAMMES,
    RAW_ETUDIANTS,
)

In [2]:
# Importation des donnees

students = pd.read_csv(RAW_ETUDIANTS,dtype={"genre": "category"})
programs = pd.read_csv(RAW_PROGRAMMES,dtype={"filiere": "string"})
inscriptions = pd.read_csv(RAW_INSCRIPTIONS,parse_dates=["date_debut", "date_annulation"], dtype={"resultat_final": "category"})
evaluations = pd.read_csv(RAW_EVALUATIONS,parse_dates=["date_echeance", "date_soumission"], dtype={"type_evaluation": "category"})
activities = pd.read_csv(RAW_ACTIVITES,parse_dates=["horodatage"], dtype={"type_activite": "category"})

In [3]:
tables = {
    
    "etudiant": students,
    "programmes":programs,
    "inscriptions":inscriptions,
    "evaluations":evaluations,
    "activites virtuelles": activities,
}


- Verification des ids pour chaque table

In [4]:
#Primary keys verification
ids = {
    "students": (students, "id_etudiant"),
    "programs": (programs, "id_programme"),
    "inscriptions": (inscriptions, "id_inscription"),
    "evaluations": (evaluations, "id_evaluation"),
    "activities": (activities, "id_activite"),
}
summary = []
for name, (table, id_col) in ids.items():
    summary.append({
        "Table": name,
        "id Column": id_col,
        "Total": len(table),
        "Null": table[id_col].isna().sum(),
        "Duplicate": table[id_col].duplicated().sum(),
        "Unique": table[id_col].is_unique,
    })
summary_df = pd.DataFrame(summary)
summary_df


,Table,id Column,Total,Null,Duplicate,Unique
0,students,id_etudiant,6392,0,0,True
1,programs,id_programme,22,0,0,True
2,inscriptions,id_inscription,7169,0,0,True
3,evaluations,id_evaluation,42312,0,0,True
4,activities,id_activite,2592987,0,0,True


In [5]:
# Verification del'unicite des programmes pour chaque etudiant dans la table inscriptions
print(inscriptions.duplicated(subset=['id_etudiant', 'id_programme']).sum())

0


- Verification des cles etrangers

In [6]:
# inscriptions avec etudiants et programmes
print("id des etudiants sont toutes dans inscriptions: ")
print(inscriptions['id_etudiant'].isin(students['id_etudiant']).sum()==inscriptions['id_etudiant'].count())

print("id des programmes sont toutes dans inscriptions: ")
print(inscriptions['id_programme'].isin(programs['id_programme']).sum()==inscriptions['id_programme'].count())

# evaluations avec inscripion
print("id des inscriptions sont toutes dans evaluations: ")
print(evaluations['id_inscription'].isin(inscriptions['id_inscription']).sum()==evaluations['id_inscription'].count())
    
# activites virtuelles avec inscriptions
print("id des inscriptions sont toutes dans activites virtuelles: ")
print(activities['id_inscription'].isin(inscriptions['id_inscription']).sum()==activities['id_inscription'].count())


id des etudiants sont toutes dans inscriptions: 
True
id des programmes sont toutes dans inscriptions: 
True
id des inscriptions sont toutes dans evaluations: 
True
id des inscriptions sont toutes dans activites virtuelles: 
True


In [7]:
# Inspection inverse, s'il ya des valeurs qui n'existent pas comme cle etrangers

print('Les ids qui existent dans la table etudiants mais non dans la table inscriptions')
print(len(set(students['id_etudiant'])-set(inscriptions['id_etudiant'])))

print('Les ids qui existent dans la table programmes mais non dans la table inscriptions')
print(len(set(programs['id_programme'])-set(inscriptions['id_programme'])))

print('Les ids qui existent dans la table inscriptions mais non dans la table evaluations')
print(len(set(inscriptions['id_inscription'])-set(evaluations['id_inscription'])))

print('Les ids qui existent dans la table inscriptions mais non dans la table activites virtuelles')
print(len(set(inscriptions['id_inscription'])-set(activities['id_inscription'])))

Les ids qui existent dans la table etudiants mais non dans la table inscriptions
0
Les ids qui existent dans la table programmes mais non dans la table inscriptions
0
Les ids qui existent dans la table inscriptions mais non dans la table evaluations
887
Les ids qui existent dans la table inscriptions mais non dans la table activites virtuelles
0


- Verification des valeurs manquantes

In [8]:
missing =[]
for name,table in tables.items():
    print(f'Table: {name}')
    for col in table.columns:
        missing.append({
            "Column": col,
            "Missing Values": table[col].isna().sum()
        })
    missing_df = pd.DataFrame(missing)
    print(missing_df)
    missing=[]


Table: etudiant
               Column  Missing Values
0         id_etudiant               0
1               genre               0
2              region               0
3  situation_handicap               0
4                 age               0
Table: programmes
            Column  Missing Values
0     id_programme               0
1  niveau_scolaire               0
2   annee_scolaire               0
3         semestre               0
4          filiere              22
Table: inscriptions
                        Column  Missing Values
0               id_inscription               0
1                  id_etudiant               0
2                 id_programme               0
3                   date_debut               1
4              date_annulation            5438
5               resultat_final               0
6  nb_inscriptions_precedentes               0
Table: evaluations
            Column  Missing Values
0    id_evaluation               0
1   id_inscription               0
2  type_

In [9]:
inscriptions[inscriptions['date_debut'].isna()]

,id_inscription,id_etudiant,id_programme,date_debut,date_annulation,resultat_final,nb_inscriptions_precedentes
3476,15968,604398,DDD_2013J,NaT,2023-08-31,Abandon,0


In [10]:
pd.crosstab(inscriptions['resultat_final'],inscriptions['date_annulation'].notna())

date_annulation,False,True
resultat_final,,
Abandon,14,1727
Echec,1651,4
Reussite,3773,0


In [11]:
pd.crosstab(evaluations['note'].isna(),evaluations['type_evaluation'])

type_evaluation,Devoir,Examen,Quiz
note,,,
False,23938,1208,17118
True,48,0,0


In [12]:
missing_eval = evaluations[evaluations['note'].isna()]
missing_eval = missing_eval.merge(
    inscriptions[
        ["id_inscription","resultat_final","date_debut","date_annulation"]
    ],
    on='id_inscription',
    how='left'
)
print(missing_eval["resultat_final"].value_counts(dropna='False'))

resultat_final
Echec       22
Abandon     16
Reussite    10
Name: count, dtype: int64


- Verification de la validite des valeurs pour chaque variable des tables

In [13]:
#Pour les variables numeriques
for name, table in tables.items():
    print(f'La table {name} \n')
    print(table.describe())
    print("\n")

La table etudiant 

        id_etudiant       region  situation_handicap          age
count  6.392000e+03  6392.000000         6392.000000  6392.000000
mean   7.137215e+05     5.844493            0.094493    14.731227
std    5.585034e+05     3.594477            0.292536     1.845641
min    2.362900e+04     0.000000            0.000000    11.000000
25%    5.086540e+05     3.000000            0.000000    13.000000
50%    5.903240e+05     6.000000            0.000000    15.000000
75%    6.468945e+05     9.000000            0.000000    16.000000
max    2.698535e+06    11.000000            1.000000    20.000000


La table programmes 

       id_programme niveau_scolaire annee_scolaire    semestre filiere
count            22              22             22          22       0
unique           22               8              2           2       0
top       AAA_2013J             3AC      2024/2025  Semestre 1     NaN
freq              1               4             13          13     NaN


La ta

In [14]:
#Pour les variables categoriques
cat = ['genre','type_evaluation','semestre','resultat_final','type_activite','niveau_scolaire','annee_scolaire']
for name,table in tables.items():
    print(f'La table {name}:')
    for col in cat:
        if col in table.columns:
            print(f'la variable {col} contient les valeurs : {table[col].unique()} \n')


La table etudiant:
la variable genre contient les valeurs : ['M', 'F']
Categories (2, str): ['F', 'M'] 

La table programmes:
la variable semestre contient les valeurs : <ArrowStringArray>
['Semestre 1', 'Semestre 2']
Length: 2, dtype: str 

la variable niveau_scolaire contient les valeurs : <ArrowStringArray>
['1AC', '3AC', '2AC', '1BAC', 'TC', '2BAC', 'Prepa Postbac', 'Concours']
Length: 8, dtype: str 

la variable annee_scolaire contient les valeurs : <ArrowStringArray>
['2023/2024', '2024/2025']
Length: 2, dtype: str 

La table inscriptions:
la variable resultat_final contient les valeurs : ['Reussite', 'Echec', 'Abandon']
Categories (3, str): ['Abandon', 'Echec', 'Reussite'] 

La table evaluations:
la variable type_evaluation contient les valeurs : ['Devoir', 'Quiz', 'Examen']
Categories (3, str): ['Devoir', 'Examen', 'Quiz'] 

La table activites virtuelles:
la variable type_activite contient les valeurs : ['Contenu de cours', 'Ressource', 'Page d'accueil', 'Forum', 'Glossaire', .

In [15]:
#Pour les variables temporelles
annees = programs['annee_scolaire'].str.split('/', expand=True)
programs['date_debut_programme'] = pd.to_datetime(
    np.where(
        programs['semestre'] == 'Semestre 1',
        annees[0] + '-09-01',
        annees[1] + '-01-01',
    )
)

inscriptions_dates = inscriptions.merge(
    programs[['id_programme', 'date_debut_programme']],
    on='id_programme',
    how='left',
)

#Verification temporelle sur inscriptions
err_annul_debut = inscriptions_dates[
    inscriptions_dates['date_annulation'].notna()
    & (inscriptions_dates['date_annulation'] < inscriptions_dates['date_debut'])
]
err_annul_prog = inscriptions_dates[
    inscriptions_dates['date_annulation'].notna()
    & (inscriptions_dates['date_annulation'] < inscriptions_dates['date_debut_programme'])
]
debut_avant_prog = inscriptions_dates[
    inscriptions_dates['date_debut'].notna()
    & (inscriptions_dates['date_debut'] < inscriptions_dates['date_debut_programme'])
]

print(f"Inscriptions avec date_annulation < date_debut : {len(err_annul_debut)}")
print(f"Inscriptions avec date_annulation < date_debut_programme (T_0) : {len(err_annul_prog)}")
print(f"Inscriptions avec date_debut < date_debut_programme (T_0) : {len(debut_avant_prog)}")




Inscriptions avec date_annulation < date_debut : 0
Inscriptions avec date_annulation < date_debut_programme (T_0) : 127
Inscriptions avec date_debut < date_debut_programme (T_0) : 7118


In [16]:
#Verification temporelle sur evaluations
eval_dates = evaluations.merge(
    inscriptions_dates[['id_inscription', 'date_debut', 'date_debut_programme', 'date_annulation']],
    on='id_inscription',
    how='left',
)

print(f"Evaluations avec date_soumission < date_debut : {(eval_dates['date_soumission'] < eval_dates['date_debut']).sum()}")
print(f"Evaluations avec date_echeance < date_debut : {(eval_dates['date_echeance'] < eval_dates['date_debut']).sum()}")
print(f"Evaluations avec date_soumission < date_debut_programme (T_0) : {(eval_dates['date_soumission'] < eval_dates['date_debut_programme']).sum()}")
print(f"Evaluations avec date_echeance < date_debut_programme (T_0) : {(eval_dates['date_echeance'] < eval_dates['date_debut_programme']).sum()}")
print(f"Evaluations avec date_soumission > date_annulation : {(eval_dates['date_soumission'] > eval_dates['date_annulation']).sum()}")

Evaluations avec date_soumission < date_debut : 29
Evaluations avec date_echeance < date_debut : 9
Evaluations avec date_soumission < date_debut_programme (T_0) : 516
Evaluations avec date_echeance < date_debut_programme (T_0) : 0
Evaluations avec date_soumission > date_annulation : 118


In [17]:
# Verification temporelle sur activites_virtuelles
act_dates = activities.merge(
    inscriptions_dates[['id_inscription', 'date_debut', 'date_debut_programme', 'date_annulation']],
    on='id_inscription',
    how='left',
)

print(f"Activites avec horodatage < date_debut : {(act_dates['horodatage'] < act_dates['date_debut']).sum()}")
print(f"Activites avec horodatage < date_debut_programme (T_0) : {(act_dates['horodatage'] < act_dates['date_debut_programme']).sum()}")
print(f"Activites avec horodatage > date_annulation : {(act_dates['horodatage'] > act_dates['date_annulation']).sum()}")

Activites avec horodatage < date_debut : 0
Activites avec horodatage < date_debut_programme (T_0) : 169607
Activites avec horodatage > date_annulation : 5801


## Conclusion

- **Intégrité des clés et couverture :** Toutes les clés primaires et étrangères des 5 tables sont valides, sans doublons ni valeurs nulles. Contrairement à l'ancienne version des données, toutes les 7 169 inscriptions disposent d'au moins une trace dans `activites_virtuelles` (0 inscription inactive), tandis que 887 inscriptions (~12,4 %) n'ont aucune évaluation enregistrée dans `evaluations`.
- **Colonnes et valeurs manquantes :**
  - La colonne `filiere` de la table `programmes` est 100 % nulle (22/22) et sera supprimée.
  - Dans `evaluations`, les 686 `date_echeance` manquantes (liées aux examens) et les 48 `note` manquantes (toutes sur des devoirs) sont structurelles et conservées.
  - La seule inscription ayant `date_debut` manquante (`id_inscription = 15968`) possède une `date_annulation` (2023-08-31) antérieure au début du programme ($T_0 = \text{2023-09-01}$).
- **Anomalies sur `inscriptions` à exclure :**
  - **127 inscriptions** présentent une `date_annulation` antérieure à la date officielle de début du programme ($T_0$, `date_debut_programme`). Ces étudiants s'étant désistés avant le début des cours, ils sont exclus du périmètre de modélisation.
  - **4 inscriptions** ont un `resultat_final` égal à `Echec` tout en ayant une `date_annulation` renseignée (incohérence label/date) et sont également exclues.
  - 14 abandons ne disposent pas de `date_annulation`.
- **Anomalies temporelles sur `evaluations` à exclure :**
  - **29 évaluations** ont une `date_soumission < date_debut` et **9 évaluations** ont une `date_echeance < date_debut` ; elles seront exclues lors du nettoyage.